<a href="https://colab.research.google.com/github/kwabenaafoakwafrempong-cell/STC-Ghana-Predictive-Maintenance-/blob/main/STC_Ghana_ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Mount Google Drive & Install Required Libraries

from google.colab import drive
drive.mount('/content/drive')

# Install any libraries not pre-installed in Colab
!pip install shap xgboost imbalanced-learn plotly --quiet

print(" Google Drive mounted and libraries installed successfully.")


# Import All Libraries
# Core
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Scikit-learn: Pre-processing & Splitting
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV,
    cross_val_score, cross_validate
)
from sklearn.preprocessing import StandardScaler, label_binarize

# Scikit-learn: Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

#  Scikit-learn: Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, balanced_accuracy_score,
    cohen_kappa_score
)
from sklearn.calibration import calibration_curve

# XGBoost
import xgboost as xgb

# SHAP Explainability
import shap

# Imbalanced-learn
from imblearn.over_sampling import SMOTE

print("All libraries imported successfully.")
print(f"   scikit-learn : imported")
print(f"   xgboost      : {xgb.__version__}")
print(f"   shap         : {shap.__version__}")



# STEP 1: Data Loading & Initial Inspection

print("=" * 70)
print("  STEP 0 - DATA LOADING & INITIAL INSPECTION")
print("=" * 70)

# File path (update folder name if different)
DATA_PATH = '/content/drive/MyDrive/STC Ghana dataset/stc_ghana_synthetic_fleet_maintenance.csv'

df = pd.read_csv(DATA_PATH)

# Basic shape & dtypes
print(f"\n  Dataset loaded from:\n    {DATA_PATH}")
print(f"\n  Shape          : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n  Column names   :\n    {list(df.columns)}")
print(f"\n  Data types     :\n{df.dtypes.to_string()}")

# Missing values
print(f"\n🔍  Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "     No missing values found.")

# Duplicate rows
dupes = df.duplicated().sum()
print(f"\n  Duplicate rows : {dupes}")

# Target variable
print(f"\n  Target column  : 'maintenance_required'")
print(f"\n    Class counts   :\n{df['maintenance_required'].value_counts().rename({0:'0 — No Maintenance', 1:'1 — Maintenance Required'}).to_string()}")
print(f"\n    Class proportions (%):\n{(df['maintenance_required'].value_counts(normalize=True)*100).round(2).rename({0:'0 — No Maintenance', 1:'1 — Maintenance Required'}).to_string()}")

# Descriptive statistics
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(f"\n Descriptive Statistics (numerical features):")
print(df.describe().round(3).to_string())

# Identify column types
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != 'maintenance_required']

print(f"\n  Categorical columns ({len(cat_cols)}) : {cat_cols}")
print(f"  Numerical columns  ({len(num_cols)}) : {num_cols}")

for c in cat_cols:
    print(f"\n    '{c}' unique values:\n{df[c].value_counts().to_string()}")


# STEP 2: Exploratory Data Analysis (EDA)

print("=" * 70)
print("  STEP 1 — EXPLORATORY DATA ANALYSIS (EDA)")
print("=" * 70)

# 2A. Target class distribution (pie + bar)
class_counts = df['maintenance_required'].value_counts().sort_index()
class_labels = ['No Maintenance (0)', 'Maintenance Required (1)']

fig_target = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'bar'}]],
    subplot_titles=['Class Distribution (Pie)', 'Class Distribution (Bar)']
)
fig_target.add_trace(go.Pie(
    labels=class_labels, values=class_counts.values,
    marker_colors=['#636EFA', '#EF553B'],
    textinfo='percent+label', hole=0.35
), row=1, col=1)
fig_target.add_trace(go.Bar(
    x=class_labels, y=class_counts.values,
    marker_color=['#636EFA', '#EF553B'],
    text=class_counts.values, textposition='outside'
), row=1, col=2)
fig_target.update_layout(
    title_text='Target Variable — maintenance_required Class Distribution',
    template='plotly_white', height=420, showlegend=False
)
fig_target.show()

# 2B. Numerical feature distributions
n = len(num_cols)
cols_per_row = 3
rows_needed  = -(-n // cols_per_row)

fig_hist = make_subplots(
    rows=rows_needed, cols=cols_per_row,
    subplot_titles=num_cols
)
for idx, col in enumerate(num_cols):
    r = idx // cols_per_row + 1
    c = idx %  cols_per_row + 1
    fig_hist.add_trace(go.Histogram(
        x=df[col], name=col,
        marker_color='#636EFA', opacity=0.75,
        nbinsx=30
    ), row=r, col=c)
fig_hist.update_layout(
    title_text='Distribution of Numerical Features',
    template='plotly_white',
    height=300 * rows_needed,
    showlegend=False
)
fig_hist.show()

# 2C. Box plots - numerical features by target class
fig_box = make_subplots(
    rows=rows_needed, cols=cols_per_row,
    subplot_titles=num_cols
)
for idx, col in enumerate(num_cols):
    r = idx // cols_per_row + 1
    c = idx %  cols_per_row + 1
    for cls, clr, lbl in [(0, '#636EFA', 'No Maint.'),
                           (1, '#EF553B', 'Maint. Req.')]:
        fig_box.add_trace(go.Box(
            y=df.loc[df['maintenance_required'] == cls, col],
            name=lbl, marker_color=clr,
            showlegend=(idx == 0),
            legendgroup=str(cls)
        ), row=r, col=c)
fig_box.update_layout(
    title_text='Box Plots — Numerical Features by Target Class',
    template='plotly_white',
    height=300 * rows_needed,
    boxmode='group'
)
fig_box.show()

# 2D. Categorical features vs target
for col in cat_cols:
    ct = df.groupby([col, 'maintenance_required']).size().reset_index(name='count')
    ct['label'] = ct['maintenance_required'].map(
        {0: 'No Maintenance', 1: 'Maintenance Required'}
    )
    fig_cat = px.bar(
        ct, x=col, y='count', color='label',
        barmode='group',
        color_discrete_map={'No Maintenance': '#636EFA',
                            'Maintenance Required': '#EF553B'},
        title=f'Categorical Feature: {col} — Count by Target Class',
        template='plotly_white', height=420
    )
    fig_cat.show()

# 2E. Correlation heatmap (numerical features)
corr_df   = df[num_cols + ['maintenance_required']].corr().round(3)
fig_corr  = go.Figure(go.Heatmap(
    z=corr_df.values,
    x=corr_df.columns.tolist(),
    y=corr_df.index.tolist(),
    colorscale='RdBu', zmid=0,
    text=corr_df.values.round(2),
    texttemplate='%{text}',
    colorbar=dict(title='Pearson r')
))
fig_corr.update_layout(
    title='Correlation Matrix — Numerical Features + Target',
    template='plotly_white', height=580,
    xaxis=dict(tickangle=45)
)
fig_corr.show()

# 2F. Target correlation bar chart (features vs target)
target_corr = corr_df['maintenance_required'].drop('maintenance_required').sort_values()
fig_tcorr = go.Figure(go.Bar(
    x=target_corr.values,
    y=target_corr.index.tolist(),
    orientation='h',
    marker_color=['#EF553B' if v > 0 else '#636EFA' for v in target_corr.values],
    text=[f"{v:.3f}" for v in target_corr.values],
    textposition='outside'
))
fig_tcorr.update_layout(
    title='Feature Correlation with Target (maintenance_required)',
    xaxis_title='Pearson Correlation Coefficient',
    template='plotly_white', height=480,
    margin=dict(l=200)
)
fig_tcorr.show()

print("\n EDA charts rendered.")


# STEP 3: Data Pre-processing

print("=" * 70)
print("  STEP 2 — DATA PRE-PROCESSING")
print("=" * 70)

# 3A. Separate features and target
X = df.drop('maintenance_required', axis=1)
y = df['maintenance_required']

print(f"\n  Features shape : {X.shape}")
print(f"  Target shape   : {y.shape}")

# 3B. One-hot encode categorical columns
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)
print(f"\n  After one-hot encoding:")
print(f"    Columns before : {X.shape[1]}  →  Columns after : {X_encoded.shape[1]}")
print(f"    New columns    : {[c for c in X_encoded.columns if c not in X.columns]}")

# 3C. Stratified 80:20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"\n  Train / Test split (80:20, stratified):")
print(f"    Training   : {X_train.shape[0]} samples")
print(f"      Class 0  : {(y_train==0).sum()}  ({(y_train==0).mean()*100:.1f}%)")
print(f"      Class 1  : {(y_train==1).sum()}  ({(y_train==1).mean()*100:.1f}%)")
print(f"    Test       : {X_test.shape[0]} samples")
print(f"      Class 0  : {(y_test==0).sum()}  ({(y_test==0).mean()*100:.1f}%)")
print(f"      Class 1  : {(y_test==1).sum()}  ({(y_test==1).mean()*100:.1f}%)")

# 3D. Feature scaling — StandardScaler (fit on TRAIN only)
# Only numerical columns are scaled; OHE columns remain 0/1
scaler = StandardScaler()

# Scale only numeric columns (not OHE dummies)
all_feature_cols = X_train.columns.tolist()
ohe_cols         = [c for c in all_feature_cols if c not in num_cols]

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols]  = scaler.transform(X_test[num_cols])

feature_names = all_feature_cols
print(f"\n  StandardScaler applied (fit on training set only — no data leakage).")
print(f"  Scaled features  ({len(num_cols)}) : {num_cols}")
print(f"  OHE columns kept ({len(ohe_cols)}) : {ohe_cols}")
print(f"\n  Final feature set ({len(feature_names)} features):\n  {feature_names}")

# Convert to numpy arrays
X_tr = X_train_scaled.values
X_te = X_test_scaled.values
y_tr = y_train.values
y_te = y_test.values

print("\n Pre-processing complete.")



# STEP 4: Class Imbalance Analysis

print("=" * 70)
print("  STEP 3 — CLASS IMBALANCE ANALYSIS")
print("=" * 70)

neg_cnt = (y_tr == 0).sum()
pos_cnt = (y_tr == 1).sum()
spw     = neg_cnt / pos_cnt

print(f"\n  Training set class counts:")
print(f"    Class 0 (No Maintenance)      : {neg_cnt}")
print(f"    Class 1 (Maintenance Required): {pos_cnt}")
print(f"    Imbalance ratio (neg:pos)     : {spw:.2f}:1")
print(f"\n  Strategy selected:")
print(f"    • Logistic Regression  → class_weight='balanced'")
print(f"    • Random Forest        → class_weight='balanced'")
print(f"    • XGBoost              → scale_pos_weight={spw:.2f}")
print(f"    • All models use Stratified K-Fold to preserve class ratio across folds")

# Visualise imbalance
fig_imb = go.Figure()
fig_imb.add_trace(go.Bar(
    x=['No Maintenance (0)', 'Maintenance Required (1)'],
    y=[neg_cnt, pos_cnt],
    marker_color=['#636EFA', '#EF553B'],
    text=[f"{neg_cnt} ({neg_cnt/(neg_cnt+pos_cnt)*100:.1f}%)",
          f"{pos_cnt} ({pos_cnt/(neg_cnt+pos_cnt)*100:.1f}%)"],
    textposition='outside', width=0.4
))
fig_imb.update_layout(
    title='Training Set Class Distribution — Class Imbalance',
    xaxis_title='Class', yaxis_title='Sample Count',
    template='plotly_white', height=420
)
fig_imb.show()

print("\n Class imbalance analysis complete.")



# STEP 5: Hyperparameter Tuning

print("=" * 70)
print("  STEP 4 — HYPERPARAMETER TUNING (5-Fold Stratified CV)")
print("=" * 70)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 5A. LOGISTIC REGRESSION — GridSearchCV

print("\n" + "─" * 60)
print("  4A. Logistic Regression — GridSearchCV")
print("─" * 60)

lr_param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100]
}
lr_base = LogisticRegression(
    penalty='l2',
    class_weight='balanced',
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)
lr_search = GridSearchCV(
    estimator=lr_base,
    param_grid=lr_param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    refit=True,
    verbose=0,
    return_train_score=True
)
lr_search.fit(X_tr, y_tr)
lr_best = lr_search.best_estimator_

print(f"\n  Best hyperparameters : {lr_search.best_params_}")
print(f"  Best CV F1-Score     : {lr_search.best_score_:.4f}")

# GridSearchCV results
lr_cv_df = pd.DataFrame(lr_search.cv_results_)[
    ['param_C', 'mean_test_score', 'std_test_score', 'rank_test_score']
].sort_values('rank_test_score')
print(f"\n  Full grid search results:\n{lr_cv_df.to_string(index=False)}")

# Plot LR tuning curve
lr_Cs    = [r['C'] for r in lr_search.cv_results_['params']]
lr_means = lr_search.cv_results_['mean_test_score']
lr_stds  = lr_search.cv_results_['std_test_score']

fig_lr_tune = go.Figure()
fig_lr_tune.add_trace(go.Scatter(
    x=[str(c) for c in lr_Cs], y=lr_means,
    error_y=dict(type='data', array=lr_stds, visible=True),
    mode='lines+markers',
    marker=dict(color='#636EFA', size=9),
    line=dict(color='#636EFA', width=2),
    name='CV F1 (mean ± std)'
))
fig_lr_tune.update_layout(
    title='Logistic Regression — GridSearchCV: C vs CV F1-Score',
    xaxis_title='Regularisation Parameter C',
    yaxis_title='Mean CV F1-Score',
    template='plotly_white', height=420
)
fig_lr_tune.show()

# 5B. RANDOM FOREST — RandomizedSearchCV

print("\n" + "─" * 60)
print("  4B. Random Forest — RandomizedSearchCV (n_iter=30)")
print("─" * 60)

rf_param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2']
}
rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=30,
    cv=skf,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    refit=True,
    verbose=0,
    return_train_score=True
)
rf_search.fit(X_tr, y_tr)
rf_best = rf_search.best_estimator_

print(f"\n  Best hyperparameters : {rf_search.best_params_}")
print(f"  Best CV F1-Score     : {rf_search.best_score_:.4f}")

# Plot RF tuning results (top 15 by rank)
rf_cv_df = pd.DataFrame(rf_search.cv_results_)[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
].sort_values('rank_test_score').head(15)
print(f"\n  Top-15 random search results:\n{rf_cv_df.to_string(index=False)}")

rf_sorted = pd.DataFrame(rf_search.cv_results_).sort_values('rank_test_score')
fig_rf_tune = go.Figure()
fig_rf_tune.add_trace(go.Scatter(
    y=rf_sorted['mean_test_score'].values,
    x=list(range(1, len(rf_sorted) + 1)),
    error_y=dict(type='data', array=rf_sorted['std_test_score'].values, visible=True),
    mode='lines+markers',
    marker=dict(color='#00CC96', size=7),
    line=dict(color='#00CC96', width=2),
    name='CV F1 by iteration rank'
))
fig_rf_tune.update_layout(
    title='Random Forest — RandomizedSearchCV: Results Ranked by CV F1',
    xaxis_title='Rank (1 = Best)',
    yaxis_title='Mean CV F1-Score',
    template='plotly_white', height=420
)
fig_rf_tune.show()


#  5C. XGBOOST — RandomizedSearchCV

print("\n" + "─" * 60)
print("  4C. XGBoost — RandomizedSearchCV (n_iter=30)")
print("─" * 60)

xgb_param_dist = {
    'learning_rate':    [0.01, 0.05, 0.1, 0.2, 0.3],
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 5, 7, 9],
    'subsample':        [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'reg_alpha':        [0, 0.1, 0.5],
    'reg_lambda':       [1, 1.5, 2.0]
}
xgb_base = xgb.XGBClassifier(
    scale_pos_weight=spw,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1
)
xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=30,
    cv=skf,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    refit=True,
    verbose=0,
    return_train_score=True
)
xgb_search.fit(X_tr, y_tr)
xgb_best = xgb_search.best_estimator_

print(f"\n  Best hyperparameters : {xgb_search.best_params_}")
print(f"  Best CV F1-Score     : {xgb_search.best_score_:.4f}")

xgb_sorted = pd.DataFrame(xgb_search.cv_results_).sort_values('rank_test_score')
fig_xgb_tune = go.Figure()
fig_xgb_tune.add_trace(go.Scatter(
    y=xgb_sorted['mean_test_score'].values,
    x=list(range(1, len(xgb_sorted) + 1)),
    error_y=dict(type='data', array=xgb_sorted['std_test_score'].values, visible=True),
    mode='lines+markers',
    marker=dict(color='#EF553B', size=7),
    line=dict(color='#EF553B', width=2),
    name='CV F1 by iteration rank'
))
fig_xgb_tune.update_layout(
    title='XGBoost — RandomizedSearchCV: Results Ranked by CV F1',
    xaxis_title='Rank (1 = Best)',
    yaxis_title='Mean CV F1-Score',
    template='plotly_white', height=420
)
fig_xgb_tune.show()

# Collect best models dict
models = {
    'Logistic Regression': lr_best,
    'Random Forest':       rf_best,
    'XGBoost':             xgb_best
}
best_params_log = {
    'Logistic Regression': str(lr_search.best_params_),
    'Random Forest':       str(rf_search.best_params_),
    'XGBoost':             str(xgb_search.best_params_)
}

print("\n Hyperparameter tuning complete. Best estimators ready.")



# STEP 6: 5-Fold Stratified Cross-Validation

print("=" * 70)
print("  STEP 5 — 5-FOLD STRATIFIED CROSS-VALIDATION (Training Set)")
print("=" * 70)

colors        = ['#636EFA', '#00CC96', '#EF553B']
cv_results    = {}
cv_metrics    = ['roc_auc', 'f1', 'precision', 'recall', 'accuracy']
cv_metric_labels = ['ROC-AUC', 'F1-Score', 'Precision', 'Recall', 'Accuracy']

for name, model in models.items():
    scores = cross_validate(
        model, X_tr, y_tr,
        cv=skf,
        scoring=cv_metrics,
        n_jobs=-1,
        return_train_score=False
    )
    cv_results[name] = {
        metric: {
            'mean': scores[f'test_{metric}'].mean(),
            'std':  scores[f'test_{metric}'].std(),
            'vals': scores[f'test_{metric}']
        }
        for metric in cv_metrics
    }
    print(f"\n  {name}:")
    for m, ml in zip(cv_metrics, cv_metric_labels):
        mean = cv_results[name][m]['mean']
        std  = cv_results[name][m]['std']
        print(f"    {ml:<12} : {mean:.4f} ± {std:.4f}")

# CV ROC-AUC bar chart
names_list = list(cv_results.keys())
means_auc  = [cv_results[n]['roc_auc']['mean'] for n in names_list]
stds_auc   = [cv_results[n]['roc_auc']['std']  for n in names_list]

fig_cv_auc = go.Figure()
fig_cv_auc.add_trace(go.Bar(
    x=names_list, y=means_auc,
    error_y=dict(type='data', array=stds_auc, visible=True),
    marker_color=colors,
    text=[f"{m:.4f}" for m in means_auc],
    textposition='outside', width=0.4
))
fig_cv_auc.update_layout(
    title='5-Fold Stratified CV — ROC-AUC (Mean ± Std) per Model',
    yaxis_title='ROC-AUC', yaxis_range=[0, 1.12],
    template='plotly_white', height=450
)
fig_cv_auc.show()

# CV multi-metric grouped bar chart
fig_cv_multi = go.Figure()
for i, name in enumerate(names_list):
    fig_cv_multi.add_trace(go.Bar(
        x=cv_metric_labels,
        y=[cv_results[name][m]['mean'] for m in cv_metrics],
        error_y=dict(
            type='data',
            array=[cv_results[name][m]['std'] for m in cv_metrics],
            visible=True
        ),
        name=name, marker_color=colors[i]
    ))
fig_cv_multi.update_layout(
    title='5-Fold Stratified CV — All Metrics (Mean ± Std)',
    barmode='group', yaxis_range=[0, 1.15],
    yaxis_title='Score', template='plotly_white', height=500
)
fig_cv_multi.show()

#  Box plots of per-fold scores
fig_cv_box = go.Figure()
for i, name in enumerate(names_list):
    fig_cv_box.add_trace(go.Box(
        y=cv_results[name]['roc_auc']['vals'],
        name=name, marker_color=colors[i],
        boxmean='sd'
    ))
fig_cv_box.update_layout(
    title='5-Fold CV — Per-Fold ROC-AUC Distribution',
    yaxis_title='ROC-AUC', yaxis_range=[0, 1.1],
    template='plotly_white', height=450
)
fig_cv_box.show()

print("\n Cross-validation complete.")



# STEP 7: Final Model Training & Test-Set Evaluation

print("=" * 70)
print("  STEP 6 — FINAL MODEL TRAINING & TEST-SET EVALUATION")
print("=" * 70)

results_rows = []
cm_data      = {}
prob_dict    = {}

for name, model in models.items():
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    prob_dict[name] = y_prob

    acc  = accuracy_score(y_te, y_pred)
    bacc = balanced_accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    auc  = roc_auc_score(y_te, y_prob)
    mcc  = matthews_corrcoef(y_te, y_pred)
    kap  = cohen_kappa_score(y_te, y_pred)
    cm   = confusion_matrix(y_te, y_pred)
    cm_data[name] = cm

    results_rows.append({
        'Model':             name,
        'Accuracy':          round(acc,  4),
        'Balanced_Accuracy': round(bacc, 4),
        'Precision':         round(prec, 4),
        'Recall':            round(rec,  4),
        'F1_Score':          round(f1,   4),
        'ROC_AUC':           round(auc,  4),
        'MCC':               round(mcc,  4),
        'Cohen_Kappa':       round(kap,  4),
        'CV_AUC_Mean':       round(cv_results[name]['roc_auc']['mean'], 4),
        'CV_AUC_Std':        round(cv_results[name]['roc_auc']['std'],  4),
        'CV_F1_Mean':        round(cv_results[name]['f1']['mean'], 4),
        'CV_F1_Std':         round(cv_results[name]['f1']['std'],  4),
        'Best_Params':       best_params_log[name]
    })

    print(f"\n{'═'*55}")
    print(f"  {name}")
    print(f"{'═'*55}")
    print(f"  Accuracy          = {acc:.4f}")
    print(f"  Balanced Accuracy = {bacc:.4f}")
    print(f"  Precision         = {prec:.4f}")
    print(f"  Recall            = {rec:.4f}")
    print(f"  F1-Score          = {f1:.4f}")
    print(f"  ROC-AUC           = {auc:.4f}")
    print(f"  MCC               = {mcc:.4f}")
    print(f"  Cohen's Kappa     = {kap:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"    TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"    FN={cm[1,0]}  TP={cm[1,1]}")
    print(f"\n{classification_report(y_te, y_pred, target_names=['No Maintenance','Maintenance Required'])}")

results_df = pd.DataFrame(results_rows)

# Performance grouped bar chart
metric_labels = ['Accuracy', 'Balanced Acc.', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
metric_keys   = ['Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC']

fig_perf = go.Figure()
for i, name in enumerate(results_df['Model']):
    vals = [results_df.loc[results_df['Model']==name, k].values[0] for k in metric_keys]
    fig_perf.add_trace(go.Bar(
        name=name, x=metric_labels, y=vals,
        marker_color=colors[i],
        text=[f"{v:.3f}" for v in vals], textposition='outside'
    ))
fig_perf.update_layout(
    title='Model Performance Comparison — Test Set Metrics',
    barmode='group', yaxis_range=[0, 1.18],
    yaxis_title='Score', template='plotly_white', height=520
)
fig_perf.show()

# Confusion matrices (3 subplots)
cm_labels = ['No Maint.', 'Maint. Req.']
fig_cm = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Logistic Regression', 'Random Forest', 'XGBoost'],
    horizontal_spacing=0.08
)
for i, name in enumerate(['Logistic Regression', 'Random Forest', 'XGBoost']):
    cm = cm_data[name]

    # Flip so top-left = TN (standard orientation: actual=rows, predicted=cols)
    annot = [[f"TN\n{cm[0,0]}", f"FP\n{cm[0,1]}"],
             [f"FN\n{cm[1,0]}", f"TP\n{cm[1,1]}"]]
    fig_cm.add_trace(go.Heatmap(
        z=cm,
        x=cm_labels, y=cm_labels,
        text=[[f"TN: {cm[0,0]}", f"FP: {cm[0,1]}"],
              [f"FN: {cm[1,0]}", f"TP: {cm[1,1]}"]],
        texttemplate='%{text}',
        colorscale='Blues',
        showscale=(i == 2)
    ), row=1, col=i+1)

    # Axis labels
    fig_cm.update_xaxes(title_text='Predicted', row=1, col=i+1)
    fig_cm.update_yaxes(title_text='Actual',    row=1, col=1)

fig_cm.update_layout(
    title='Confusion Matrices — LR | RF | XGBoost (Test Set)',
    height=420, template='plotly_white'
)
fig_cm.show()

print("\n Test-set evaluation complete.")


# STEP 8: ROC Curves & Precision-Recall Curves

print("=" * 70)
print("  STEP 7 — ROC CURVES & PRECISION-RECALL CURVES")
print("=" * 70)

fig_roc = go.Figure()
fig_pr  = go.Figure()

for i, (name, model) in enumerate(models.items()):
    y_prob = prob_dict[name]

    #  ROC
    fpr, tpr, thresholds_roc = roc_curve(y_te, y_prob)
    auc_val = roc_auc_score(y_te, y_prob)
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{name}  (AUC = {auc_val:.4f})',
        line=dict(color=colors[i], width=2.5)
    ))

    #  Precision-Recall
    prec_arr, rec_arr, _ = precision_recall_curve(y_te, y_prob)
    ap = average_precision_score(y_te, y_prob)
    fig_pr.add_trace(go.Scatter(
        x=rec_arr, y=prec_arr, mode='lines',
        name=f'{name}  (AP = {ap:.4f})',
        line=dict(color=colors[i], width=2.5)
    ))

# Random-chance baseline on ROC
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier  (AUC = 0.50)',
    line=dict(dash='dash', color='grey', width=1.5)
))

# No-skill baseline on PR (= class prevalence)
baseline_pr = pos_cnt / (neg_cnt + pos_cnt)
fig_pr.add_trace(go.Scatter(
    x=[0, 1], y=[baseline_pr, baseline_pr], mode='lines',
    name=f'No-Skill Baseline  (AP = {baseline_pr:.2f})',
    line=dict(dash='dash', color='grey', width=1.5)
))

fig_roc.update_layout(
    title='ROC Curves — LR vs RF vs XGBoost',
    xaxis_title='False Positive Rate (1 – Specificity)',
    yaxis_title='True Positive Rate (Sensitivity / Recall)',
    legend=dict(x=0.55, y=0.05),
    template='plotly_white', height=520
)
fig_roc.show()

fig_pr.update_layout(
    title='Precision-Recall Curves — LR vs RF vs XGBoost',
    xaxis_title='Recall (Sensitivity)',
    yaxis_title='Precision',
    legend=dict(x=0.4, y=0.95),
    template='plotly_white', height=520
)
fig_pr.show()

print("\n ROC and Precision-Recall curves rendered.")


# STEP 9: SHAP Explainability

print("=" * 70)
print("  STEP 8 — SHAP EXPLAINABILITY")
print("  (Global Feature Importance + Local Explanations + Dependence Plot)")
print("=" * 70)

# 9A. Compute SHAP values for all three models

# Logistic Regression - LinearExplainer
lr_explainer = shap.LinearExplainer(lr_best, X_tr)
lr_shap_raw  = lr_explainer.shap_values(X_te)
lr_shap      = np.array(lr_shap_raw)
if lr_shap.ndim == 3:
    lr_shap = lr_shap[:, :, 1]
print(f"  LR  SHAP → shape: {lr_shap.shape}")

# Random Forest - TreeExplainer (handles list or 3D for class 1)
rf_explainer = shap.TreeExplainer(rf_best)
rf_shap_raw  = rf_explainer.shap_values(X_te)
if isinstance(rf_shap_raw, list):
    rf_shap = np.array(rf_shap_raw[1])
elif np.array(rf_shap_raw).ndim == 3:
    rf_shap = np.array(rf_shap_raw)[:, :, 1]
else:
    rf_shap = np.array(rf_shap_raw)
print(f"  RF  SHAP → shape: {rf_shap.shape}")

# XGBoost - TreeExplainer
xgb_explainer = shap.TreeExplainer(xgb_best)
xgb_shap_raw  = xgb_explainer.shap_values(X_te)
if isinstance(xgb_shap_raw, list):
    xgb_shap = np.array(xgb_shap_raw[1])
elif np.array(xgb_shap_raw).ndim == 3:
    xgb_shap = np.array(xgb_shap_raw)[:, :, 1]
else:
    xgb_shap = np.array(xgb_shap_raw)
print(f"  XGB SHAP → shape: {xgb_shap.shape}")

# 9B. Build global SHAP summary table
shap_summary = pd.DataFrame({'Feature': feature_names})
shap_summary['LR_Mean_AbsSHAP']  = np.abs(lr_shap).mean(axis=0)
shap_summary['RF_Mean_AbsSHAP']  = np.abs(rf_shap).mean(axis=0)
shap_summary['XGB_Mean_AbsSHAP'] = np.abs(xgb_shap).mean(axis=0)
shap_summary['Mean_Across_Models'] = shap_summary[
    ['LR_Mean_AbsSHAP', 'RF_Mean_AbsSHAP', 'XGB_Mean_AbsSHAP']
].mean(axis=1)
shap_summary = shap_summary.sort_values(
    'Mean_Across_Models', ascending=False
).reset_index(drop=True)
shap_summary['Rank'] = range(1, len(shap_summary) + 1)

print(f"\n  Top 10 Features by Mean |SHAP| Across All Models:")
print(shap_summary[['Rank', 'Feature', 'LR_Mean_AbsSHAP',
                     'RF_Mean_AbsSHAP', 'XGB_Mean_AbsSHAP',
                     'Mean_Across_Models']].head(10).to_string(index=False))

# 9C. Per-model global SHAP bar charts (horizontal)
shap_vals_dict = {
    'Logistic Regression': (lr_shap,  colors[0]),
    'Random Forest':       (rf_shap,  colors[1]),
    'XGBoost':             (xgb_shap, colors[2])
}

for name, (sv, clr) in shap_vals_dict.items():
    mean_abs = np.abs(sv).mean(axis=0)
    order    = np.argsort(mean_abs)

    fig_g = go.Figure()
    fig_g.add_trace(go.Bar(
        y=[feature_names[i] for i in order],
        x=[mean_abs[i]      for i in order],
        orientation='h',
        marker_color=clr,
        text=[f"{mean_abs[i]:.4f}" for i in order],
        textposition='outside'
    ))
    fig_g.update_layout(
        title=f'Global Feature Importance — Mean |SHAP|  ({name})',
        xaxis_title='Mean |SHAP Value|',
        template='plotly_white', height=520,
        margin=dict(l=220)
    )
    fig_g.show()

# 9D. Cross-model SHAP comparison (grouped horizontal bar)
sorted_feats = shap_summary['Feature'].tolist()[::-1]

fig_comp = go.Figure()
for col, lbl, clr in [
    ('LR_Mean_AbsSHAP',  'Logistic Regression', colors[0]),
    ('RF_Mean_AbsSHAP',  'Random Forest',        colors[1]),
    ('XGB_Mean_AbsSHAP', 'XGBoost',              colors[2])
]:
    vals = [shap_summary.loc[shap_summary['Feature']==f, col].values[0]
            for f in sorted_feats]
    fig_comp.add_trace(go.Bar(
        y=sorted_feats, x=vals,
        name=lbl, orientation='h', marker_color=clr
    ))
fig_comp.update_layout(
    title='SHAP Feature Importance Comparison — All Three Models',
    barmode='group', xaxis_title='Mean |SHAP Value|',
    template='plotly_white', height=640, margin=dict(l=220)
)
fig_comp.show()

# 9E. Local SHAP — single vehicle explanations (XGBoost)
xgb_probs = xgb_best.predict_proba(X_te)[:, 1]
high_idx  = int(np.argmax(xgb_probs))
low_idx   = int(np.argmin(xgb_probs))

print(f"\n  Local explanation — XGBoost:")
print(f"    High-risk vehicle → test index {high_idx},  P(maintenance) = {xgb_probs[high_idx]:.4f}")
print(f"    Low-risk  vehicle → test index {low_idx},   P(maintenance) = {xgb_probs[low_idx]:.4f}")

for label, idx_val in [('High-Risk Vehicle', high_idx), ('Low-Risk Vehicle', low_idx)]:
    sv    = xgb_shap[idx_val]
    order = np.argsort(np.abs(sv))
    clrs  = ['#EF553B' if sv[i] > 0 else '#636EFA' for i in order]

    fig_local = go.Figure()
    fig_local.add_trace(go.Bar(
        y=[feature_names[i] for i in order],
        x=[sv[i]            for i in order],
        orientation='h',
        marker_color=clrs,
        text=[f"{sv[i]:+.4f}" for i in order],
        textposition='outside'
    ))
    fig_local.update_layout(
        title=f'Local SHAP — {label}  (XGBoost, P = {xgb_probs[idx_val]:.4f})\n'
              f'Red bars = push TOWARD maintenance;  Blue bars = push AWAY',
        xaxis_title='SHAP Value (contribution to model output)',
        template='plotly_white', height=520, margin=dict(l=220)
    )
    fig_local.show()

# 9F. SHAP Dependence Plot — top 2 features
top_feat    = shap_summary['Feature'].iloc[0]
second_feat = shap_summary['Feature'].iloc[1]
top_i       = feature_names.index(top_feat)
sec_i       = feature_names.index(second_feat)

fig_dep = go.Figure()
fig_dep.add_trace(go.Scatter(
    x=X_te[:, top_i],
    y=xgb_shap[:, top_i],
    mode='markers',
    marker=dict(
        color=X_te[:, sec_i],
        colorscale='Viridis',
        size=6, opacity=0.78,
        colorbar=dict(title=second_feat)
    ),
    text=[f"{top_feat}={X_te[j, top_i]:.2f}, "
          f"{second_feat}={X_te[j, sec_i]:.2f}"
          for j in range(len(X_te))],
    hoverinfo='text'
))
fig_dep.update_layout(
    title=(f'SHAP Dependence Plot — <b>{top_feat}</b>  '
           f'(coloured by <b>{second_feat}</b>)  [XGBoost]'),
    xaxis_title=top_feat,
    yaxis_title=f'SHAP Value for {top_feat}',
    template='plotly_white', height=520
)
fig_dep.show()

print("\n SHAP analysis complete.")


# STEP 10: Model Calibration (Reliability Diagrams)

print("=" * 70)
print("  STEP 9 — MODEL CALIBRATION (RELIABILITY DIAGRAMS)")
print("=" * 70)

fig_cal = go.Figure()

# Perfect calibration diagonal
fig_cal.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines', name='Perfect Calibration',
    line=dict(dash='dash', color='grey', width=1.5)
))

for i, (name, model) in enumerate(models.items()):
    y_prob = prob_dict[name]
    prob_true, prob_pred = calibration_curve(
        y_te, y_prob, n_bins=10, strategy='uniform'
    )
    fig_cal.add_trace(go.Scatter(
        x=prob_pred, y=prob_true,
        mode='lines+markers', name=name,
        line=dict(color=colors[i], width=2.5),
        marker=dict(size=9, symbol='circle')
    ))

    # Brier score (lower = better calibration)
    brier = np.mean((y_prob - y_te) ** 2)
    print(f"  {name}  →  Brier Score = {brier:.4f}")

fig_cal.update_layout(
    title='Reliability Diagrams — Model Calibration (Probability Accuracy)',
    xaxis_title='Mean Predicted Probability',
    yaxis_title='Fraction of Positives (Observed Frequency)',
    xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
    template='plotly_white', height=520,
    legend=dict(x=0.02, y=0.96)
)
fig_cal.show()

print("\n  Interpretation: A model perfectly calibrated sits on the diagonal.")
print("  Points ABOVE → under-confident; Points BELOW → over-confident.")
print("\n Calibration analysis complete.")


# STEP 11: Results Summary & Save to Google Drive

print("=" * 70)
print("  STEP 10 — FINAL RESULTS SUMMARY & SAVE TO GOOGLE DRIVE")
print("=" * 70)

# Full performance table
summary_cols = ['Model', 'Accuracy', 'Balanced_Accuracy', 'Precision',
                'Recall', 'F1_Score', 'ROC_AUC', 'MCC', 'Cohen_Kappa',
                'CV_AUC_Mean', 'CV_AUC_Std', 'CV_F1_Mean', 'CV_F1_Std']

print("\n📊  FULL MODEL PERFORMANCE COMPARISON TABLE")
print("─" * 90)
print(results_df[summary_cols].to_string(index=False))

# Winner by F1 and ROC-AUC
best_f1  = results_df.loc[results_df['F1_Score'].idxmax(), 'Model']
best_auc = results_df.loc[results_df['ROC_AUC'].idxmax(), 'Model']
print(f"\n  Best Model by F1-Score  : {best_f1}")
print(f"  Best Model by ROC-AUC   : {best_auc}")

# Top 10 predictive features
print(f"\n🔍  TOP 10 PREDICTIVE FEATURES (Mean |SHAP| Across All Models)")
print("─" * 60)
print(shap_summary[['Rank', 'Feature', 'LR_Mean_AbsSHAP',
                     'RF_Mean_AbsSHAP', 'XGB_Mean_AbsSHAP',
                     'Mean_Across_Models']].head(10).to_string(index=False))

# Final radar chart (multi-metric comparison)
radar_metrics = ['Accuracy', 'Balanced_Accuracy', 'Precision',
                 'Recall', 'F1_Score', 'ROC_AUC']
radar_labels  = ['Accuracy', 'Bal. Accuracy', 'Precision',
                 'Recall', 'F1-Score', 'ROC-AUC']

fig_radar = go.Figure()
for i, row in results_df.iterrows():
    vals = [row[m] for m in radar_metrics]
    fig_radar.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=radar_labels + [radar_labels[0]],
        fill='toself', opacity=0.35,
        line=dict(color=colors[i], width=2),
        marker=dict(color=colors[i], size=7),
        name=row['Model']
    ))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Radar Chart — Multi-Metric Model Comparison (Test Set)',
    template='plotly_white', height=530
)
fig_radar.show()

# Save CSV outputs to Google Drive
SAVE_DIR = '/content/drive/MyDrive/STC Ghana dataset/'

results_path = SAVE_DIR + 'ml_results_summary.csv'
shap_path    = SAVE_DIR + 'shap_values_summary.csv'

results_df.to_csv(results_path, index=False)
shap_summary.to_csv(shap_path, index=False)

print(f"\n  ml_results_summary.csv   saved → {results_path}  ({len(results_df)} models)")
print(f"  shap_values_summary.csv  saved → {shap_path}  ({len(shap_summary)} features)")

print("\n" + "=" * 70)
print("    PIPELINE COMPLETE — ALL STEPS SUCCESSFULLY EXECUTED")
print("=" * 70)
print("""
  Summary of what was produced:

  ✔ STEP 0  — Dataset loaded & inspected
  ✔ STEP 1  — EDA: distributions, box plots, categorical plots, correlation
  ✔ STEP 2  — Pre-processing: OHE, 80:20 stratified split, StandardScaler
  ✔ STEP 3  — Class imbalance quantified & visualised (scale_pos_weight)
  ✔ STEP 4  — Hyperparameter tuning: GridSearchCV (LR), RandomizedSearchCV (RF, XGB)
  ✔ STEP 5  — 5-Fold Stratified CV: ROC-AUC, F1, Precision, Recall, Accuracy
  ✔ STEP 6  — Test-set evaluation: Accuracy, Balanced Accuracy, Precision,
               Recall, F1, ROC-AUC, MCC, Cohen's Kappa, Confusion Matrices
  ✔ STEP 7  — ROC Curves & Precision-Recall Curves
  ✔ STEP 8  — SHAP: Global (per-model + cross-model), Local (high/low-risk),
               Dependence Plot
  ✔ STEP 9  — Model Calibration (Reliability Diagrams + Brier Scores)
  ✔ STEP 10 — Radar chart, full summary table, outputs saved to Google Drive

""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Google Drive mounted and libraries installed successfully.
All libraries imported successfully.
   scikit-learn : imported
   xgboost      : 3.4.1
   shap         : 0.52.0
  STEP 0 - DATA LOADING & INITIAL INSPECTION

  Dataset loaded from:
    /content/drive/MyDrive/STC Ghana dataset/stc_ghana_synthetic_fleet_maintenance.csv

  Shape          : 1000 rows × 13 columns

  Column names   :
    ['vehicle_age_years', 'total_mileage_km', 'engine_hours', 'days_since_last_maintenance', 'previous_faults_12m', 'component_condition_index', 'tyre_condition_score', 'brake_condition_score', 'operational_intensity_kmday', 'driver_experience_years', 'route_type', 'last_maintenance_type', 'maintenance_required']

  Data types     :
vehicle_age_years              float64
total_mileage_km                 int64
engine_hours                     int64
days_since_last_maintenance


 EDA charts rendered.
  STEP 2 — DATA PRE-PROCESSING

  Features shape : (1000, 12)
  Target shape   : (1000,)

  After one-hot encoding:
    Columns before : 12  →  Columns after : 14
    New columns    : ['route_type_Peri-Urban', 'route_type_Urban', 'last_maintenance_type_Preventive', 'last_maintenance_type_Reactive']

  Train / Test split (80:20, stratified):
    Training   : 800 samples
      Class 0  : 680  (85.0%)
      Class 1  : 120  (15.0%)
    Test       : 200 samples
      Class 0  : 170  (85.0%)
      Class 1  : 30  (15.0%)

  StandardScaler applied (fit on training set only — no data leakage).
  Scaled features  (10) : ['vehicle_age_years', 'total_mileage_km', 'engine_hours', 'days_since_last_maintenance', 'previous_faults_12m', 'component_condition_index', 'tyre_condition_score', 'brake_condition_score', 'operational_intensity_kmday', 'driver_experience_years']
  OHE columns kept (4) : ['route_type_Peri-Urban', 'route_type_Urban', 'last_maintenance_type_Preventive', 'las


 Class imbalance analysis complete.
  STEP 4 — HYPERPARAMETER TUNING (5-Fold Stratified CV)

────────────────────────────────────────────────────────────
  4A. Logistic Regression — GridSearchCV
────────────────────────────────────────────────────────────

  Best hyperparameters : {'C': 1}
  Best CV F1-Score     : 0.6800

  Full grid search results:
 param_C  mean_test_score  std_test_score  rank_test_score
   1.000         0.680035        0.050776                1
  10.000         0.675991        0.057614                2
 100.000         0.675991        0.057614                2
   0.100         0.662418        0.052178                4
   0.010         0.632205        0.021912                5
   0.001         0.611713        0.025853                6



────────────────────────────────────────────────────────────
  4B. Random Forest — RandomizedSearchCV (n_iter=30)
────────────────────────────────────────────────────────────

  Best hyperparameters : {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
  Best CV F1-Score     : 0.6621

  Top-15 random search results:
                                                                                                          params  mean_test_score  std_test_score  rank_test_score
{'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}         0.662076        0.063530                1
  {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 10}         0.657674        0.066914                2
 {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None}         0.657247  


────────────────────────────────────────────────────────────
  4C. XGBoost — RandomizedSearchCV (n_iter=30)
────────────────────────────────────────────────────────────

  Best hyperparameters : {'subsample': 0.6, 'reg_lambda': 2.0, 'reg_alpha': 0.5, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.6}
  Best CV F1-Score     : 0.6936


...


 ROC and Precision-Recall curves rendered.
  STEP 8 — SHAP EXPLAINABILITY
  (Global Feature Importance + Local Explanations + Dependence Plot)
  LR  SHAP → shape: (200, 14)
  RF  SHAP → shape: (200, 14)
  XGB SHAP → shape: (200, 14)

  Top 10 Features by Mean |SHAP| Across All Models:
 Rank                     Feature  LR_Mean_AbsSHAP  RF_Mean_AbsSHAP  XGB_Mean_AbsSHAP  Mean_Across_Models
    1           vehicle_age_years         1.471001         0.094666          1.160194            0.908620
    2 days_since_last_maintenance         0.965130         0.054535          0.781006            0.600223
    3   component_condition_index         0.702329         0.038072          0.627495            0.455965
    4        tyre_condition_score         0.752986         0.033303          0.487171            0.424487
    5         previous_faults_12m         0.795140         0.023046          0.427492            0.415226
    6       brake_condition_score         0.780080         0.027711          


  Local explanation — XGBoost:
    High-risk vehicle → test index 33,  P(maintenance) = 0.9829
    Low-risk  vehicle → test index 122,   P(maintenance) = 0.0007



 SHAP analysis complete.
  STEP 9 — MODEL CALIBRATION (RELIABILITY DIAGRAMS)
  Logistic Regression  →  Brier Score = 0.1233
  Random Forest  →  Brier Score = 0.1046
  XGBoost  →  Brier Score = 0.1173



  Interpretation: A model perfectly calibrated sits on the diagonal.
  Points ABOVE → under-confident; Points BELOW → over-confident.

 Calibration analysis complete.
  STEP 10 — FINAL RESULTS SUMMARY & SAVE TO GOOGLE DRIVE

📊  FULL MODEL PERFORMANCE COMPARISON TABLE
──────────────────────────────────────────────────────────────────────────────────────────
              Model  Accuracy  Balanced_Accuracy  Precision  Recall  F1_Score  ROC_AUC    MCC  Cohen_Kappa  CV_AUC_Mean  CV_AUC_Std  CV_F1_Mean  CV_F1_Std
Logistic Regression     0.825             0.8010     0.4510  0.7667    0.5679   0.8943 0.4931       0.4673       0.9421      0.0239      0.6800     0.0508
      Random Forest     0.855             0.6814     0.5200  0.4333    0.4727   0.8469 0.3916       0.3895       0.9147      0.0148      0.6621     0.0635
            XGBoost     0.830             0.7216     0.4474  0.5667    0.5000   0.8588 0.4033       0.3993       0.9298      0.0112      0.6936     0.0604

  Best Model by F1-


  ml_results_summary.csv   saved → /content/drive/MyDrive/STC Ghana dataset/ml_results_summary.csv  (3 models)
  shap_values_summary.csv  saved → /content/drive/MyDrive/STC Ghana dataset/shap_values_summary.csv  (14 features)

    PIPELINE COMPLETE — ALL STEPS SUCCESSFULLY EXECUTED

  Summary of what was produced:
  
  ✔ STEP 0  — Dataset loaded & inspected
  ✔ STEP 1  — EDA: distributions, box plots, categorical plots, correlation
  ✔ STEP 2  — Pre-processing: OHE, 80:20 stratified split, StandardScaler
  ✔ STEP 3  — Class imbalance quantified & visualised (scale_pos_weight)
  ✔ STEP 4  — Hyperparameter tuning: GridSearchCV (LR), RandomizedSearchCV (RF, XGB)
  ✔ STEP 5  — 5-Fold Stratified CV: ROC-AUC, F1, Precision, Recall, Accuracy
  ✔ STEP 6  — Test-set evaluation: Accuracy, Balanced Accuracy, Precision,
               Recall, F1, ROC-AUC, MCC, Cohen's Kappa, Confusion Matrices
  ✔ STEP 7  — ROC Curves & Precision-Recall Curves
  ✔ STEP 8  — SHAP: Global (per-model + cross-model),